# Phase 1: ClinVar Cleaning & Labeling
# المرحلة 1: تنظيف بيانات كلين فار وتصنيفها

We clean the raw ClinVar data to keep only high-quality,
clinically classified variants for training our models.


In [ ]:
# Load raw ClinVar data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_raw = pd.read_csv('../data/raw/clinvar/variant_summary.txt.gz',
                     sep='\t', low_memory=False)

print('=' * 60)
print('RAW CLINVAR DATA')
print('=' * 60)
print(f'Total rows:    {len(df_raw):,}')
print(f'Total columns: {len(df_raw.columns)}')
print(f'Memory usage:  {df_raw.memory_usage(deep=True).sum() / 1e6:.0f} MB')


## Step 1: Filter by Variant Type

We keep only Single Nucleotide Variants (SNVs) because our models
and features (from dbNSFP) are designed for this type.


In [ ]:
# Step 1: Keep only SNVs
before = len(df_raw)
df = df_raw[df_raw['Type'] == 'single nucleotide variant'].copy()
after = len(df)

print("Filter: Type == 'single nucleotide variant'")
print(f'Before: {before:>12,}')
print(f'After:  {after:>12,}')
print(f'Removed: {before - after:>11,} ({(before-after)/before*100:.1f}%)')


## Step 2: Filter Clinical Significance
## الخطوة 2: فلترة التصنيف السريري

We keep only variants with clear clinical classifications:
- **Pathogenic** and **Likely pathogenic** → Label = 1 (disease-causing)
- **Benign** and **Likely benign** → Label = 0 (safe)

We exclude VUS (Uncertain significance) and conflicting interpretations.


In [ ]:
# Step 2: Keep clear clinical labels only
keep_labels = ['Pathogenic', 'Likely pathogenic', 'Benign', 'Likely benign']

before = len(df)
df = df[df['ClinicalSignificance'].isin(keep_labels)]
after = len(df)

print('Filter: Keep only clear clinical labels')
print(f'Before: {before:>12,}')
print(f'After:  {after:>12,}')
print(f'Removed: {before - after:>11,}')
print()
print('Label distribution:')
print(df['ClinicalSignificance'].value_counts().to_string())


In [ ]:
# Create binary label
df['label'] = df['ClinicalSignificance'].map({
    'Pathogenic': 1, 'Likely pathogenic': 1,
    'Benign': 0, 'Likely benign': 0
})

pathogenic_count = int((df['label'] == 1).sum())
benign_count = int((df['label'] == 0).sum())
print('Binary label distribution:')
print(f'  Pathogenic (1): {pathogenic_count:>10,} ({pathogenic_count/len(df)*100:.1f}%)')
print(f'  Benign (0):     {benign_count:>10,} ({benign_count/len(df)*100:.1f}%)')


## Step 3: Filter by Review Quality
## الخطوة 3: فلترة جودة المراجعة

ClinVar assigns review stars (0-4) based on evidence quality.
We keep only variants with at least 1 star (reviewed by submitter).


In [ ]:
# Step 3: Parse review stars and filter
# (Logic adapted from src/clinvar_cleaning.py)

def _normalize_text(value):
    if pd.isna(value):
        return ''
    return str(value).strip().lower()

def extract_review_stars(value):
    text = _normalize_text(value)
    if not text:
        return 0
    if 'practice guideline' in text:
        return 4
    if 'reviewed by expert panel' in text:
        return 3
    if 'multiple submitters' in text and 'no conflicts' in text and 'criteria provided' in text:
        return 2
    if 'criteria provided' in text and (
        'single submitter' in text
        or 'conflicting interpretations' in text
        or 'multiple submitters' in text
    ):
        return 1
    return 0

if "ReviewStatus" in df.columns:
    df['review_stars'] = df['ReviewStatus'].map(extract_review_stars).astype('int8')
else:
    df['review_stars'] = 0

print("Review stars distribution (before filtering):")
print(df['review_stars'].value_counts().sort_index().to_string())

before = len(df)
df = df[df['review_stars'] >= 1].copy()
after = len(df)

print('\nFilter: review_stars >= 1')
print(f'Before: {before:>12,}')
print(f'After:  {after:>12,}')
print(f'Removed: {before - after:>11,} ({(before-after)/before*100:.1f}%)')


## Step 4: Standardize & Remove Duplicates
## الخطوة 4: توحيد الصيغة وحذف المكرر

- Standardize chromosome naming (1-22, X, Y)
- Create variant key: chr:pos:ref:alt
- Remove duplicate variant keys
- Remove variants with conflicting labels (same variant, different label)


In [ ]:
# Standardize and deduplicate
# (Logic adapted from src/clinvar_cleaning.py)

def normalize_chromosome(value):
    text = _normalize_text(value)
    if not text:
        return None
    if text.startswith('chr'):
        text = text[3:]
    text = text.upper()
    if text == '23':
        return 'X'
    if text == '24':
        return 'Y'
    if text.isdigit():
        return str(int(text))
    if text in {'X', 'Y'}:
        return text
    return text

def normalize_allele_series(series):
    out = series.fillna('').astype(str).str.strip().str.upper()
    return out.replace({'': '', 'NA': '', 'NAN': '', 'NONE': '', 'NULL': '', '.': '', '-': ''})

allowed_chromosomes = set([str(i) for i in range(1, 23)] + ['X', 'Y'])

df['chr'] = df['Chromosome'].map(normalize_chromosome)
before_chr = len(df)
df = df[df['chr'].isin(allowed_chromosomes)].copy()
print(f'Rows after chromosome normalization/filter: {before_chr:,} -> {len(df):,}')

pos_vcf = pd.to_numeric(df['PositionVCF'], errors='coerce') if 'PositionVCF' in df.columns else pd.Series(pd.NA, index=df.index, dtype='float64')
pos_start = pd.to_numeric(df['Start'], errors='coerce') if 'Start' in df.columns else pd.Series(pd.NA, index=df.index, dtype='float64')
df['pos'] = pos_vcf.where(pos_vcf.notna(), pos_start)

ref_vcf = normalize_allele_series(df['ReferenceAlleleVCF']) if 'ReferenceAlleleVCF' in df.columns else pd.Series('', index=df.index)
ref_raw = normalize_allele_series(df['ReferenceAllele']) if 'ReferenceAllele' in df.columns else pd.Series('', index=df.index)
alt_vcf = normalize_allele_series(df['AlternateAlleleVCF']) if 'AlternateAlleleVCF' in df.columns else pd.Series('', index=df.index)
alt_raw = normalize_allele_series(df['AlternateAllele']) if 'AlternateAllele' in df.columns else pd.Series('', index=df.index)

df['ref'] = ref_vcf.where(ref_vcf.ne(''), ref_raw)
df['alt'] = alt_vcf.where(alt_vcf.ne(''), alt_raw)
df['gene'] = df['GeneSymbol'].fillna('').astype(str).str.strip()

before_clean = len(df)
df = df[
    df['pos'].notna()
    & df['ref'].ne('')
    & df['alt'].ne('')
    & df['gene'].ne('')
    & df['ref'].str.len().eq(1)
    & df['alt'].str.len().eq(1)
].copy()
df['pos'] = df['pos'].astype('int64')
print(f'Rows after required-field cleanup: {before_clean:,} -> {len(df):,}')

df['variant_key'] = df['chr'] + ':' + df['pos'].astype(str) + ':' + df['ref'] + ':' + df['alt']

if 'MolecularConsequence' not in df.columns:
    df['MolecularConsequence'] = pd.NA
if 'PhenotypeIDS' not in df.columns:
    df['PhenotypeIDS'] = pd.NA

conflict_key_mask = df.groupby("variant_key")["label"].nunique() > 1
conflict_keys = set(conflict_key_mask[conflict_key_mask].index.tolist())
conflicting_labels_removed = int(df['variant_key'].isin(conflict_keys).sum())

if conflict_keys:
    df = df[~df['variant_key'].isin(conflict_keys)].copy()

duplicate_rows_found = int(df.duplicated(subset=['variant_key'], keep='first').sum())
before_dedup = len(df)
df = df.drop_duplicates(subset=['variant_key'], keep='first').copy()
duplicates_removed = before_dedup - len(df)

keep_columns = [
    'chr', 'pos', 'ref', 'alt', 'gene', 'label', 'review_stars',
    'variant_key', 'ClinicalSignificance', 'MolecularConsequence', 'PhenotypeIDS'
]
df = df[keep_columns].copy()

print(f'Duplicate rows found:        {duplicate_rows_found:,}')
print(f'Conflicting label rows removed: {conflicting_labels_removed:,}')
print(f'Final row count after dedup: {len(df):,}')


## Step 5: Missense Filtering Note
## الخطوة 5: ملاحظة عن فلترة الطفرات

ClinVar's variant_summary.txt does not include a MolecularConsequence column.
Missense filtering is applied later during the merge with dbNSFP (Phase 4),
because dbNSFP only contains missense variants by design.


In [ ]:
# Save cleaned data
df.to_parquet('../data/intermediate/clinvar_labeled_clean.parquet', index=False)

pathogenic_count = int((df['label'] == 1).sum())
benign_count = int((df['label'] == 0).sum())

print('=' * 60)
print('FINAL CLEANED CLINVAR DATASET')
print('=' * 60)
print(f'Total variants:  {len(df):,}')
print(f'Pathogenic (1):  {pathogenic_count:,}')
print(f'Benign (0):      {benign_count:,}')
print(f"Unique genes:    {df['gene'].nunique():,}")
print(f"Chromosomes:     {sorted(df['chr'].unique())}")
print()
print('Saved to: data/intermediate/clinvar_labeled_clean.parquet')


In [ ]:
# Visualization: Class distribution
import os
os.makedirs('../results/figures', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
counts = df['ClinicalSignificance'].value_counts()
colors = ['#e74c3c', '#e67e22', '#27ae60', '#2ecc71']
counts.plot(kind='bar', ax=axes[0], color=colors[:len(counts)])
axes[0].set_title('Clinical Significance Distribution', fontsize=14)
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Top 15 genes
top_genes = df['gene'].value_counts().head(15)
top_genes.plot(kind='barh', ax=axes[1], color='#3498db')
axes[1].set_title('Top 15 Genes by Variant Count', fontsize=14)
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../results/figures/clinvar_cleaning_summary.png', dpi=150, bbox_inches='tight')
plt.show()
